# BlackCode FR 260M - Entraînement dual-T4 Kaggle (6Md) — ULTRA MAX v3

Toutes les optimisations possibles :
- Dataset 4 Go FineWeb2-HQ fra_Latn : heuristique ×50 + no-near ×2 RAM÷10 + Bloom RAM÷100 +20% + tokenizer 20M 5 min + tar
- Tokenizer 24k + 88milSMS CC BY 4.0 optionnel abréviations
- Train 6 Md dual-T4 : Muon + WSD + selective + compile max-autotune fullgraph + DDP no_sync +15-30% + fused + foreach + SwiGLU fused +5-10% + RMSNorm fused +5-10% + cudnn benchmark +5-10% + TF32 +10-20% + prefetch 2 +5-15% + bucket 10 +5% + FP16 compress +10-20% PCIe + static_graph +2-5% + CUDA Graphs +10%

Temps v3 MAX : 35 min build + 9-10h train 6Md = 12-13h total (vs 16h v2, 19h v1, 30h+ baseline)


In [ ]:
!pip install datasets tokenizers -q
!pip install lingua-language-detector -q || echo "fallback heuristique ×50"
!git clone https://github.com/Snowoo-2z/Llm-search.git /tmp/Llm-search || true
%cd /tmp/Llm-search
!ls configs/blackcode_fr_260m.json


In [ ]:
!python scripts/audit_fr_source.py --dataset epfml/FineWeb2-HQ:fra_Latn --limit 500 --sample-count 10


In [ ]:
# Build v3 MAX : Bloom + no-near + 20M chars + parallel + tar
!MAX_MB=4000 MIN_QUALITY_SCORE=0.6 OUT=data/fr_260m VOCAB=24576 SEQ_LEN=2048 TOKENIZER_CHARS=20000000 BLOOM=1 bash scripts/build_fr_optimized_v3.sh


In [ ]:
!python scripts/eval_french_purity.py --corpus-dir data/fr_260m --limit 50
!python scripts/benchmark_quality.py


In [ ]:
# Pilote 200 steps v3 MAX
!python -m blackcode.train --config configs/blackcode_fr_260m.json --data data/fr_260m --out runs/fr_260m_pilot --batch-size 2 --grad-accum 8 --seq-len 2048 --max-steps 200 --optim adafactor --lr 3e-4 --fused --foreach --checkpoint-policy selective --compile --compile-mode max-autotune --compile-fullgraph --cudnn-benchmark --tf32 --prefetch 2 --log-flops --eval-interval 100 --save-interval 100 --resume


In [ ]:
# Train complet 6 Md ULTRA MAX v3 — 9-10h dual-T4
!torchrun --nproc_per_node=2 -m blackcode.train --config configs/blackcode_fr_260m.json --data data/fr_260m --out runs/fr_260m_6b --batch-size 4 --grad-accum 8 --seq-len 2048 --max-steps 183105 --optim muon --muon-lr 0.02 --lr 6e-4 --fused --foreach --schedule wsd --stable-frac 0.8 --seq-len-warmup-frac 0.2 --checkpoint-policy selective --compile --compile-mode max-autotune --compile-fullgraph --cudnn-benchmark --tf32 --prefetch 2 --bucket-cap-mb 10 --ddp --ddp-fp16-compress --ddp-static-graph --log-flops --eval-interval 1000 --save-interval 1000 --resume


In [ ]:
# Alternative avec CUDA Graphs (static shape, après warmup)
# Phase 1 : warmup 20% sans graphs
!torchrun --nproc_per_node=2 -m blackcode.train --config configs/blackcode_fr_260m.json --data data/fr_260m --out runs/fr_260m_6b --batch-size 4 --grad-accum 8 --seq-len 2048 --max-steps 36621 --optim muon --muon-lr 0.02 --lr 6e-4 --fused --schedule wsd --stable-frac 0.8 --seq-len-warmup-frac 0.2 --checkpoint-policy selective --compile --compile-mode max-autotune --cudnn-benchmark --tf32 --prefetch 2 --bucket-cap-mb 10 --ddp --ddp-fp16-compress --ddp-static-graph --log-flops --eval-interval 1000 --save-interval 1000 --resume
# Phase 2 : static shape avec CUDA Graphs +10%
!torchrun --nproc_per_node=2 -m blackcode.train --config configs/blackcode_fr_260m.json --data data/fr_260m --out runs/fr_260m_6b --batch-size 4 --grad-accum 8 --seq-len 2048 --max-steps 183105 --optim muon --muon-lr 0.02 --lr 6e-4 --fused --schedule wsd --stable-frac 1.0 --checkpoint-policy selective --compile --compile-mode max-autotune-no-cudagraphs --cudnn-benchmark --tf32 --prefetch 2 --bucket-cap-mb 10 --ddp --ddp-fp16-compress --ddp-static-graph --cuda-graphs --log-flops --eval-interval 1000 --save-interval 1000 --resume
